# RAG Output Quality Experiment - CRD with Extended Thinking

**Purpose:** Evaluate RAG grounding quality of a Clue rules agent using Extended Thinking to observe whether answers are grounded in retrieved chunks or training data.

**Design:**
- **Experiment Type:** Completely Randomized Design (CRD)
- **Treatment:** None — fixed temperature = 1.0 (required by extended thinking)
- **Replicates:** 2 per question (each question asked twice)
- **Total Runs:** 31 questions × 2 replicates = 62 runs
- **Run Order:** Completely randomized

**Response Variables:**
- RGS = C1 × (C2 + C3 + C4 + C5) / 4
- `agent_thinking`: Claude's chain-of-thought before answering (extended thinking)
- `judge_thinking`: Judge's chain-of-thought during scoring (extended thinking)

**Workflow:**
1. Load 31 questions
2. Create design matrix: 31 questions × 2 replicates = 62 runs
3. Completely randomize run order
4. Execute experiment → CSV with responses + agent thinking
5. Score C1–C5 via LLM-as-judge (with judge extended thinking captured)
6. Calculate RGS
7. Import to JMP for analysis

# System Design
### RAG Configuration
-- Semantic Chunking based on existing headers in game instructions
-- Reranker
-- Contextualized Chunks
-- Prompt Caching


## 1. Setup and Imports

In [ ]:
# Core imports
import os
import sys
import json
import random
import time
import csv
import re
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# API clients
import voyageai

# Exception handling (still needed for answer_clue_question retry logic)
try:
    from anthropic import APITimeoutError, APIConnectionError, RateLimitError
except ImportError:
    APITimeoutError = TimeoutError
    APIConnectionError = ConnectionError
    RateLimitError = Exception

# Add parent directory to path and import shared helpers
sys.path.append("..")
import llm_helpers
from llm_helpers import chat, add_user_message, add_assistant_message, text_from_message
from rag_helpers import build_clue_retriever, tools
from scoring_helper import create_scoring_template, merge_scores

print("✓ Imports successful")

In [ ]:
# Load environment and initialize clients
load_dotenv(dotenv_path='../.env')

# VoyageAI for embeddings
v_api_key = os.getenv('VOYAGE_API_KEY')
embedding_client = voyageai.Client(api_key=v_api_key)

# Initialize shared LLM helpers (sets internal client, model, max_tokens)
model = os.getenv('MODEL_NAME')
max_tokens = int(os.getenv('MAX_TOKENS'))
THINKING_BUDGET_TOKENS = int(os.getenv('THINKING_BUDGET_TOKENS'))

llm_helpers.init(
    api_key=os.getenv('ANTHROPIC_API_KEY'),
    model_name=model,
    max_tokens=max_tokens
)

In [ ]:
# Fixed temperature for CRD experiment
# Extended thinking requires temperature = 1.0 — this is an API constraint,
# not a design choice. All 62 runs use the same temperature.
EXPERIMENT_TEMPERATURE = 1.0

print(f"✓ Temperature fixed at: {EXPERIMENT_TEMPERATURE}")
print(f"✓ Thinking budget: {THINKING_BUDGET_TOKENS} tokens per call")

## 3. RAG Setup

In [ ]:
# RAG Setup — build hybrid retriever from shared rag_helpers
# This runs the full setup pipeline: load document → chunk → contextualise
# → embed → index. Expensive (API calls) but only runs once per session.
retriever, search_clue_instructions = build_clue_retriever(
    embedding_client,
    "../References/ClueRules.md"
)
print("✓ RAG setup complete")

## 4. System Prompt

In [ ]:
# Load system prompt from JSON file
USE_GROUNDED_PROMPT = True  # ← flip to False to use systemPrompt.json

prompt = "GroundedSystemPrompt.json" if USE_GROUNDED_PROMPT else "systemPrompt.json"

with open("../AgentPrompts/" + prompt, "r") as f:
    system_prompt_data = json.load(f)

system_prompt = "\n".join(system_prompt_data["system_prompt"])

print(f"✓ System prompt loaded from AgentPrompts/{prompt}")
print(f"  Length: {len(system_prompt)} characters")
print(f"✓ Tool schema imported from rag_helpers ({len(tools)} tool(s))")

## 5. Agent with Extended Thinking and Retry Logic

**Key changes from RCBD version:**
- `temperature_value` default changed to `1.0` (required by extended thinking)
- `thinking` parameter added to every `chat()` call
- `agent_thinking` captured from the `thinking` content block in the response
- All return paths include `agent_thinking` field

In [ ]:
def answer_clue_question(user_question, temperature_value=1.0, max_iterations=5,
                          max_retries=3, retry_delay=5):
    """
    Answer a Clue rules question using RAG with extended thinking.

    Args:
        user_question (str):    The user's question about Clue
        temperature_value (float): Must be 1.0 (required by extended thinking)
        max_iterations (int):   Max tool-use cycles
        max_retries (int):      Max retry attempts on transient errors
        retry_delay (int):      Seconds between retries

    Returns:
        dict: {
            "answer": str,
            "agent_thinking": str,   # extended thinking text (or "")
            "iterations": int,
            "retries": int,
            "error": str or None,
            "tool_calls": list,
            "retrieved_chunks": str
        }
    """

    def join_chunks(chunks_list, max_chars=50000):
        joined = " ||| ".join(chunks_list)
        return joined[:max_chars]

    def extract_thinking(response):
        """Pull thinking text from response content blocks."""
        for block in response.content:
            if block.type == "thinking":
                return block.thinking
        return ""

    retry_count = 0
    iteration_count = 0   # initialised here so except blocks can always reference it

    while retry_count <= max_retries:
        try:
            messages = []
            add_user_message(messages, user_question)
            iteration_count = 0
            tool_calls = []
            retrieved_chunks = []
            last_thinking = ""

            for iteration in range(max_iterations):
                iteration_count = iteration + 1
                print(f"--- Iteration {iteration + 1} ---")

                response = chat(
                    messages=messages,
                    system=system_prompt,
                    temperature=temperature_value,
                    tools=tools,
                    timeout=120,
                    thinking={"type": "enabled", "budget_tokens": THINKING_BUDGET_TOKENS}
                )

                # Capture thinking from each turn (final turn is most informative)
                last_thinking = extract_thinking(response)
                print(f"Stop reason: {response.stop_reason}")

                if response.stop_reason == "end_turn":
                    return {
                        "answer":           text_from_message(response),
                        "agent_thinking":   last_thinking,
                        "iterations":       iteration_count,
                        "retries":          retry_count,
                        "error":            None,
                        "tool_calls":       tool_calls,
                        "retrieved_chunks": join_chunks(retrieved_chunks)
                    }

                if response.stop_reason == "tool_use":
                    add_assistant_message(messages, response)

                    for content_block in response.content:
                        if content_block.type == "tool_use":
                            tool_name   = content_block.name
                            tool_input  = content_block.input
                            tool_use_id = content_block.id

                            print(f"Tool: {tool_name} | Query: {tool_input['query']}")

                            try:
                                if tool_name == "search_clue_instructions":
                                    tool_result = search_clue_instructions(tool_input["query"])
                                else:
                                    tool_result = json.dumps({"error": f"Unknown tool: {tool_name}"})

                                print(f"Result preview: {tool_result[:150]}...")

                            except Exception as tool_error:
                                print(f"⚠️ Tool error: {str(tool_error)[:100]}")
                                tool_result = json.dumps({
                                    "error": f"Tool execution failed: {str(tool_error)}",
                                    "tool": tool_name
                                })

                            tool_calls.append({"tool": tool_name, "query": tool_input.get("query", "")})
                            retrieved_chunks.append(tool_result)

                            messages.append({
                                "role": "user",
                                "content": [{
                                    "type": "tool_result",
                                    "tool_use_id": tool_use_id,
                                    "content": tool_result
                                }]
                            })
                    continue

                # Unexpected stop reason fallback
                return {
                    "answer":           text_from_message(response),
                    "agent_thinking":   last_thinking,
                    "iterations":       iteration_count,
                    "retries":          retry_count,
                    "error":            None,
                    "tool_calls":       tool_calls,
                    "retrieved_chunks": join_chunks(retrieved_chunks)
                }

            # Max iterations reached
            return {
                "answer":           "I apologize, but I'm having trouble processing this question.",
                "agent_thinking":   last_thinking,
                "iterations":       iteration_count,
                "retries":          retry_count,
                "error":            "max_iterations_reached",
                "tool_calls":       tool_calls,
                "retrieved_chunks": join_chunks(retrieved_chunks)
            }

        except (APITimeoutError, APIConnectionError, TimeoutError, ConnectionError) as e:
            error_name = type(e).__name__.lower()
            error_type = "timeout" if "timeout" in error_name else "connection"

            if iteration_count > 0:
                print(f"❌ {error_type.capitalize()} error mid-conversation - cannot retry")
                return {
                    "answer": "", "agent_thinking": "",
                    "iterations": iteration_count, "retries": retry_count,
                    "error": f"{error_type}_mid_conversation: {str(e)}",
                    "tool_calls": tool_calls if 'tool_calls' in locals() else [],
                    "retrieved_chunks": join_chunks(retrieved_chunks if 'retrieved_chunks' in locals() else [])
                }

            retry_count += 1
            if retry_count <= max_retries:
                print(f"⚠️ {error_type.capitalize()} error (attempt {retry_count}/{max_retries+1})")
                print(f"   Waiting {retry_delay}s...")
                time.sleep(retry_delay)
            else:
                print(f"❌ Max retries exceeded")
                return {
                    "answer": "", "agent_thinking": "",
                    "iterations": 0, "retries": retry_count - 1,
                    "error": f"{error_type}_error: {str(e)}",
                    "tool_calls": [], "retrieved_chunks": ""
                }

        except RateLimitError as e:
            print(f"❌ Rate limit: {str(e)}")
            return {
                "answer": "", "agent_thinking": "",
                "iterations": 0, "retries": retry_count,
                "error": f"rate_limit_error: {str(e)}",
                "tool_calls": [], "retrieved_chunks": ""
            }

        except Exception as e:
            error_name = type(e).__name__.lower()
            if "ratelimit" in error_name or "rate_limit" in error_name:
                print(f"❌ Rate limit: {str(e)}")
                return {
                    "answer": "", "agent_thinking": "",
                    "iterations": 0, "retries": retry_count,
                    "error": f"rate_limit_error: {str(e)}",
                    "tool_calls": [], "retrieved_chunks": ""
                }
            print(f"❌ Unexpected error: {str(e)}")
            return {
                "answer": "", "agent_thinking": "",
                "iterations": 0, "retries": retry_count,
                "error": f"unexpected_error: {str(e)}",
                "tool_calls": [], "retrieved_chunks": ""
            }

print("✓ Agent function defined with extended thinking and retry logic")

## 6. Load Experimental Questions

In [ ]:
def load_questions_from_csv(csv_path):
    """
    Load questions with their types from CSV.

    Returns:
        list: [{"number": 1, "question": "...", "type": "S"}, ...]
    """
    df = pd.read_csv(csv_path)

    questions = []
    for _, row in df.iterrows():
        questions.append({
            "number":   int(row['Number']),
            "question": row['Question'],
            "type":     row['Type']
        })

    print(f"✅ Loaded {len(questions)} questions from {csv_path}")

    type_counts = df['Type'].value_counts().to_dict()
    print(f"   Question types: S={type_counts.get('S', 0)}, "
          f"M={type_counts.get('M', 0)}, F={type_counts.get('F', 0)}")

    return questions

# Load questions
questions = load_questions_from_csv("../References/ClueQuestions.csv")

## 7. Post-Experiment Helper Functions

In [ ]:
# create_scoring_template and merge_scores imported from scoring_helper (see Core Imports)
print("✓ Post-experiment helpers ready")

## 9. RUN CRD EXPERIMENT

**Before running:**
- Ensure all previous cells have run successfully
- Check that `questions` variable is loaded
- Verify API keys and credit balance

**Expected runtime:** ~5–7 minutes (62 runs × 3s delay + thinking overhead)

**CRD guarantees:**
- Each question appears exactly twice (2 replicates)
- Run order is completely randomized
- No systematic confounding with run order


In [ ]:
def run_CRD_experiment(
    questions_data,
    csv_filename="../Output/Ext_ClueRag_CRD.csv",
    replicates=2,
    delay_seconds=3
):
    """
    Run CRD experiment with extended thinking at fixed temperature=1.0.

    Args:
        questions_data (list): Questions from load_questions_from_csv()
        csv_filename (str): Output CSV file
        replicates (int): Number of times each question is asked (default: 2)
        delay_seconds (int): Delay between API calls

    Returns:
        dict: Experiment statistics
    """

    os.makedirs(os.path.dirname(csv_filename), exist_ok=True)

    # ── Step 1: Build design matrix ──────────────────────────────────────────
    print("\n" + "="*70)
    print("CREATING CRD DESIGN MATRIX")
    print("="*70)

    design_matrix = []
    for rep in range(1, replicates + 1):
        for q_data in questions_data:
            design_matrix.append({
                "question_number": q_data["number"],
                "question_text":   q_data["question"],
                "question_type":   q_data["type"],
                "replicate":       rep
            })

    print(f"Design matrix created: {len(design_matrix)} runs")
    print(f"  {len(questions_data)} questions × {replicates} replicates")

    # ── Step 2: Completely randomize run order ────────────────────────────────
    print(f"\n🔀 Randomizing all {len(design_matrix)} runs...")
    random.shuffle(design_matrix)
    print("✓ Randomization complete\n")

    # ── Step 3: Statistics tracking ───────────────────────────────────────────
    stats = {
        "start_time":      datetime.now(),
        "total_runs":      len(design_matrix),
        "successful_runs": 0,
        "failed_runs":     0,
        "by_question_type": {"S": {"successful": 0, "failed": 0},
                             "M": {"successful": 0, "failed": 0},
                             "F": {"successful": 0, "failed": 0}}
    }

    # ── Step 4: Execute experiment ────────────────────────────────────────────
    fieldnames = [
        'run_id', 'timestamp',
        'replicate', 'question_number', 'question_type', 'question_text',
        'temperature_value',
        'answer',
        'agent_thinking',                # ← NEW: agent extended thinking
        'retrieved_chunks',
        'status', 'error_message', 'iteration_count', 'retry_count',
        'tool_calls_count', 'search_queries',
        # Judge scoring columns — filled by judge_submit.py
        'C1_correctness', 'C2_coverage', 'C3_citation_presence',
        'C4_citation_valid', 'C5_clarity',
        'CorrectRefusal', 'ImproperRefusal', 'ShouldHaveRefused',
        # Judge reasoning columns — flattened from judge JSON
        'reasoning_C1', 'reasoning_C2', 'reasoning_C3',
        'reasoning_C4', 'reasoning_C5', 'reasoning_refusal',
        # Judge extended thinking — filled by judge_submit.py
        'judge_thinking',                # ← NEW: judge extended thinking
        # Admin
        'scored_by', 'scoring_notes',
        # Computed
        'RGS', 'groundedness'
    ]

    with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        print("="*70)
        print("CRD EXPERIMENT WITH EXTENDED THINKING")
        print("="*70)
        print(f"Design:      Completely Randomized Design (CRD)")
        print(f"Temperature: {EXPERIMENT_TEMPERATURE} (fixed — required by extended thinking)")
        print(f"Replicates:  {replicates} per question")
        print(f"Total runs:  {len(design_matrix)}")
        print(f"Thinking:    {THINKING_BUDGET_TOKENS} tokens budget per call")
        print(f"RGS Formula: C1 × (C2+C3+C4+C5) / 4")
        print("="*70 + "\n")

        for run_id, run_spec in enumerate(design_matrix, 1):
            question_number = run_spec["question_number"]
            question_text   = run_spec["question_text"]
            question_type   = run_spec["question_type"]
            replicate       = run_spec["replicate"]

            print(f"Run {run_id:2d}/{len(design_matrix)} | Rep={replicate} | Q{question_number:2d} | Type={question_type}")
            print(f"Q: {question_text[:70]}...")

            timestamp = datetime.now().isoformat()

            result = answer_clue_question(
                question_text,
                temperature_value=EXPERIMENT_TEMPERATURE,
                max_iterations=5,
                max_retries=3,
                retry_delay=5
            )

            if result["error"] is None:
                status = "SUCCESS"
                stats["successful_runs"] += 1
                stats["by_question_type"][question_type]["successful"] += 1
                print(f"✅ Success")
                print(f"A: {result['answer'][:70]}...")
                if result['agent_thinking']:
                    print(f"💭 Thinking: {result['agent_thinking'][:100]}...")
            else:
                status = "ERROR"
                stats["failed_runs"] += 1
                stats["by_question_type"][question_type]["failed"] += 1
                print(f"❌ Error: {result['error'][:70]}...")

            writer.writerow({
                'run_id':           run_id,
                'timestamp':        timestamp,
                'replicate':        replicate,
                'question_number':  question_number,
                'question_type':    question_type,
                'question_text':    question_text,
                'temperature_value': EXPERIMENT_TEMPERATURE,
                'answer':           result["answer"],
                'agent_thinking':   result.get("agent_thinking", ""),
                'retrieved_chunks': result.get("retrieved_chunks", ""),
                'status':           status,
                'error_message':    result["error"] or '',
                'iteration_count':  result["iterations"],
                'retry_count':      result["retries"],
                'tool_calls_count': len(result.get("tool_calls", [])),
                'search_queries':   "; ".join([tc["query"] for tc in result.get("tool_calls", [])]),
                # Judge scoring — blank at run time, filled by judge_submit.py
                'C1_correctness': '', 'C2_coverage': '',
                'C3_citation_presence': '', 'C4_citation_valid': '',
                'C5_clarity': '', 'CorrectRefusal': '',
                'ImproperRefusal': '', 'ShouldHaveRefused': '',
                'reasoning_C1': '', 'reasoning_C2': '', 'reasoning_C3': '',
                'reasoning_C4': '', 'reasoning_C5': '', 'reasoning_refusal': '',
                'judge_thinking': '',   # ← filled by judge_submit.py
                'scored_by': '', 'scoring_notes': '',
                'RGS': '', 'groundedness': ''
            })

            if run_id < len(design_matrix):
                print(f"⏳ {delay_seconds}s...\n")
                time.sleep(delay_seconds)
            else:
                print()

    # ── Final statistics ───────────────────────────────────────────────────────
    stats["end_time"] = datetime.now()
    stats["duration_minutes"] = (stats["end_time"] - stats["start_time"]).total_seconds() / 60

    print("\n" + "="*70)
    print("EXPERIMENT COMPLETE")
    print("="*70)
    print(f"Total runs:  {stats['total_runs']}")
    print(f"Successful:  {stats['successful_runs']}")
    print(f"Failed:      {stats['failed_runs']}")
    print(f"Duration:    {stats['duration_minutes']:.1f} min")

    print(f"\nBy Question Type:")
    for qtype in ["S", "M", "F"]:
        s = stats["by_question_type"][qtype]
        print(f"  Type {qtype}: {s['successful']:2d} ✓, {s['failed']:2d} ✗")

    print("="*70)
    print(f"\n📁 {csv_filename}")
    print(f"📊 Next steps:")
    print(f"   1. Run judge_submit.py to fill C1–C5, refusal flags, reasoning, and judge_thinking")
    print(f"   2. Calculate RGS = C1 × (C2+C3+C4+C5) / 4")
    print(f"   3. Import to JMP for analysis\n")

    return stats

print("✓ CRD experiment function defined")

## 10. Run the CRD Experiment

Run this cell when ready. Output goes to `../Output/Ext_ClueRag_CRD.csv`.


In [ ]:
# Generate scoring template from experiment output
create_scoring_template("../Output/Ext_ClueRag_CRD.csv")

In [ ]:
# Run the CRD experiment
stats = run_CRD_experiment(
    questions_data=questions,
    csv_filename="../Output/Ext_ClueRag_CRD.csv",
    replicates=2,
    delay_seconds=3
)